In [58]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [59]:
from langchain.agents import create_react_agent, AgentExecutor 

os.environ["SSL_CERT_FILE"] = certifi.where() 
load_dotenv() 

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")
# import os
# from getpass import getpass

# os.environ["TAVILY_API_KEY"] = getpass("Tavily API key: ")

In [60]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Try common locations for the .env file
env_file = None
for base in [Path.cwd(), Path.cwd().parent]:
    candidate = base / ".env"
    if candidate.exists():
        env_file = candidate
        break

if env_file is not None:
    load_dotenv(dotenv_path=env_file, override=True)
else:
    load_dotenv(override=True)  # fallback: search standard env locations

TAVILY_API_KEY = (os.getenv("TAVILY_API_KEY") or "").strip()

if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY is missing or empty. Add it to a .env file as: "
        "TAVILY_API_KEY=your_key_here or set os.environ['TAVILY_API_KEY'] manually."
    )

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
search_tool = TavilySearchResults(max_results=2)

In [61]:
result = search_tool.invoke("Give me the latest news on asian games") 

In [62]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [63]:
result

[{'url': 'https://asiangamestv.com/news',
  'content': 'Stay updated with the latest Asian Games news, stories and announcements. Breaking news, athlete interviews and event updates on Asian Games Hub.'},
 {'url': 'https://timesofindia.indiatimes.com/sports/asian-games',
  'content': 'The 20th Asian Games, Aichi-Nagoya 2026, will take place from September 19 to October 4 in Aichi Prefecture and Nagoya, Japan. The multi-sport spectacle will feature competitions across 43 sports, with virtual taekwondo, mixed martial arts and freestyle BMX among the new additions. The Games’ mascot, HONOHON, draws inspiration from Shachi Hoko, a traditional Japanese guardian deity.  \n  \nIndia will be represented by a 503-athlete contingent, comprising 269 men and 234 women, in addition to 246 support staff members and 19 officials. Athletics will account for the largest Indian squad with 73 athletes, while hockey, shooting and cricket will also be among the major sports for the country. [...] Sporting 

In [64]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0, #creativity parameter
    api_key=OPENAI_API_KEY
)

In [65]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she kept giving him too many bytes!', response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 13, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-a377d1b4-d6b5-4933-a1f1-3465bf1852c3-0')

In [66]:
# ==========================================
# PROMPT
# ==========================================
from langchain import hub
prompt = hub.pull("hwchase17/react") 

d:\Aashish\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [67]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [68]:
tools=[search_tool, get_weather_data]

In [69]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [70]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [80]:
response = agent_executor.invoke({
    "input": (
         "Can you provide the updates of cjp protest going at shivaji park, mumbai"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
I should first search for updates on the CJP protest at Shivaji Park in Mumbai using tavily_search_results_json, and then get the current weather for Mumbai using get_weather_data.
Action: tavily_search_results_json
Action Input: CJP protest Shivaji Park Mumbai updates[{'url': 'https://timesofindia.indiatimes.com/city/mumbai/cjp-protest-live-updates-october-2-cockroach-janta-party-abhijeet-dipke-cec-gyanesh-kumar-resignation-mumbai-election-commission/liveblog/134629972.cms', 'content': 'In Mumbai, CJP founder Abhijeet Dipke has announced a "Jail Bharo Andolan" at Shivaji Park at 4pm. The CJP had earlier planned a protest against Kumar at the venue but said Mumbai Police denied permission. Police later said it was not empowered to grant permission for a gathering at Shivaji Park in view of a 2013 Bombay High Court order and described the area as a silence zone.   \n Dipke has demanded Kumar\'s resignation and threatened to expand the CJP\'s prote

In [81]:
print(response["output"])

The CJP protest at Shivaji Park in Mumbai is ongoing, and the current weather in Mumbai is sunny with a temperature of 28°C and 59% humidity.
